In [ ]:
# train_eval.py
import os
import random
import time
import argparse
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, average_precision_score, brier_score_loss

warnings.filterwarnings("ignore")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

from core_glaucogate import (
    FundusDataset, ConvNeXtBaseModel, ConvNeXtProGlaucoGate, GlaucoGateTotalLoss
)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def compute_ece(y_true, y_prob, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]
        in_bin = (y_prob > bin_lower) & (y_prob <= bin_upper)
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(y_true[in_bin])
            avg_confidence_in_bin = np.mean(y_prob[in_bin])
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    return float(ece)

def compute_selective_metrics(y_true, y_score, y_prob, sel_scores):
    idx_sorted = np.argsort(-sel_scores)
    y_true_s = y_true[idx_sorted]
    y_score_s = y_score[idx_sorted]

    n = len(y_true)
    risks = []
    coverages = np.linspace(0.1, 1.0, 10)
    for cov in coverages:
        k = max(2, int(n * cov))
        sub_true = y_true_s[:k]
        sub_score = y_score_s[:k]
        if len(np.unique(sub_true)) >= 2:
            r = 1.0 - roc_auc_score(sub_true, sub_score)
        else:
            r = 0.5
        risks.append(r)
    aurc = float(np.mean(risks))

    k90 = max(2, int(n * 0.90))
    sub90_true = y_true_s[:k90]
    sub90_score = y_score_s[:k90]
    if len(np.unique(sub90_true)) >= 2:
        fpr, tpr, _ = roc_curve(sub90_true, sub90_score)
        spec = 1.0 - fpr
        valid = np.where(spec >= 0.95)[0]
        sens95_cov90 = float(np.max(tpr[valid])) if len(valid) > 0 else 0.0
    else:
        sens95_cov90 = 0.0
    return aurc, sens95_cov90

def compute_all_metrics(y_true, y_prob, y_logit=None, sel_scores=None, fixed_threshold=None):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_score = y_logit if y_logit is not None else y_prob

    if len(np.unique(y_true)) < 2:
        return {}

    auc = roc_auc_score(y_true, y_score)
    pauc = roc_auc_score(y_true, y_score, max_fpr=0.10)
    auprc = average_precision_score(y_true, y_score)
    ece = compute_ece(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)

    if sel_scores is not None:
        aurc, sens95_cov90 = compute_selective_metrics(y_true, y_score, y_prob, sel_scores)
    else:
        aurc, sens95_cov90 = np.nan, np.nan

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    spec = 1.0 - fpr
    valid_95sp = np.where(spec >= 0.95)[0]
    roc_sens_at_95sp = float(np.max(tpr[valid_95sp])) if len(valid_95sp) > 0 else 0.0

    valid_95sens = np.where(tpr >= 0.95)[0]
    roc_spec_at_95sens = float(np.max(spec[valid_95sens])) if len(valid_95sens) > 0 else 0.0

    if fixed_threshold is None:
        if len(valid_95sp) > 0:
            best_idx = valid_95sp[np.argmax(tpr[valid_95sp])]
            thresh_use = float(thresholds[best_idx])
        else:
            thresh_use = 0.5
    else:
        thresh_use = float(fixed_threshold)

    y_pred = (y_prob >= thresh_use).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    achieved_sens = tp / max(1, (tp + fn))
    achieved_spec = tn / max(1, (tn + fp))

    return {
        "AUC": round(float(auc), 4),
        "pAUC_90_100": round(float(pauc), 4),
        "AUPRC": round(float(auprc), 4),
        "ROC_Sens@95Sp": round(roc_sens_at_95sp, 4),
        "ROC_Spec@95Sens": round(roc_spec_at_95sens, 4),
        "Sel90Cov_Sens@95Sp": round(float(sens95_cov90), 4),
        "AURC": round(float(aurc), 4),
        "ECE": round(float(ece), 4),
        "Brier": round(float(brier), 4),
        "Frozen_Threshold": round(float(thresh_use), 5),
        "FixedThresh_Sens": round(float(achieved_sens), 4),
        "FixedThresh_Spec": round(float(achieved_spec), 4),
        "FPR_at_FixedThresh": round(float(1.0 - achieved_spec), 4),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)
    }

@torch.no_grad()
def evaluate_loader(model, loader, device, save_csv_path=None, fixed_threshold=None,
                    force_center_crop=False, use_oracle_crop=False):
    model.eval()
    is_pro = isinstance(model, ConvNeXtProGlaucoGate)
    rows = []
    loc_errors, complete_disc_flags, loc_failures = [], [], []

    for batch in loader:
        x_glob = batch["global_img"].to(device, non_blocking=True)
        x_highres = batch["highres_img"].to(device, non_blocking=True)
        od_gt_dev = batch["od_gt"].to(device, non_blocking=True)
        omega = batch["omega_mask"].to(device, non_blocking=True)

        anchor_coords = od_gt_dev if use_oracle_crop else None

        with torch.amp.autocast("cuda" if device.type == "cuda" else "cpu", enabled=(device.type == "cuda"), dtype=torch.bfloat16):
            if is_pro:
                out = model(x_glob, x_highres, force_center_crop=force_center_crop, oracle_coords=anchor_coords)
                logits = out["logit"].float()
                sel_scores = out["selection_score"].float()
                stn_np = out["stn_params"].float().cpu().numpy()
                cx_p, cy_p, s_p = stn_np[:, 0], stn_np[:, 1], stn_np[:, 2]
                mq_mean = out["m_q"].float().mean(dim=(1, 2, 3)).cpu().numpy()
                gate_mean = out["gate"].float().mean(dim=(1, 2, 3)).cpu().numpy()
                gate_resized = F.interpolate(
                    out["gate"].float().mean(dim=1, keepdim=True),
                    size=omega.shape[-2:], mode="bilinear", align_corners=False
                )
                omega_sum = omega.sum(dim=(1, 2, 3))
                gate_in_omega = torch.where(
                    omega_sum > 0,
                    (gate_resized * omega).sum(dim=(1, 2, 3)) / (omega_sum + 1e-6),
                    out["gate"].float().mean(dim=(1, 2, 3))
                ).cpu().numpy()
            else:
                out = model(x_glob)
                logits = out["logit"].float()
                sel_scores = out["selection_score"].float()
                cx_p = cy_p = s_p = mq_mean = gate_mean = gate_in_omega = np.zeros(len(batch["image_id"]))

        probs = torch.sigmoid(logits).cpu().numpy()
        logits_np = logits.cpu().numpy()
        sel_np = sel_scores.cpu().numpy()
        labels = batch["label"].numpy()
        od_gts = batch["od_gt"].numpy()

        for i in range(len(probs)):
            gt_cx, gt_cy, gt_r = od_gts[i]
            if not np.isnan(gt_cx) and is_pro:
                pred_cx_val = 0.0 if force_center_crop else float(cx_p[i])
                pred_cy_val = 0.0 if force_center_crop else float(cy_p[i])
                dist = 0.5 * np.sqrt((pred_cx_val - gt_cx)**2 + (pred_cy_val - gt_cy)**2)
                r_val = gt_r if not np.isnan(gt_r) and gt_r > 0 else 0.15
                contains_full = (abs(pred_cx_val - gt_cx) + r_val <= s_p[i]) and (abs(pred_cy_val - gt_cy) + r_val <= s_p[i])
                failed = dist > 0.20
                loc_errors.append(dist)
                complete_disc_flags.append(float(contains_full))
                loc_failures.append(float(failed))

            rows.append({
                "image_id": batch["image_id"][i],
                "patient_id": batch["patient_id"][i],
                "label": int(labels[i]),
                "logit": float(logits_np[i]),
                "prob": float(probs[i]),
                "selection_score": float(sel_np[i]),
                "mq_mean": float(mq_mean[i]),
                "gate_mean": float(gate_mean[i]),
                "gate_corrupted_region": float(gate_in_omega[i]),
                "pred_cx": float(cx_p[i]), "pred_cy": float(cy_p[i]), "pred_s": float(s_p[i])
            })

    pred_df = pd.DataFrame(rows)
    metrics = compute_all_metrics(
        pred_df["label"].values, pred_df["prob"].values,
        y_logit=pred_df["logit"].values, sel_scores=pred_df["selection_score"].values,
        fixed_threshold=fixed_threshold
    )
    if save_csv_path:
        pred_df.to_csv(save_csv_path, index=False)

    if loc_errors:
        metrics["OD_Center_Err_Pct"] = round(float(np.mean(loc_errors) * 100.0), 2)
        metrics["OD_Full_Crop_Pct"] = round(float(np.mean(complete_disc_flags) * 100.0), 2)
        metrics["OD_Fail_Rate_Pct"] = round(float(np.mean(loc_failures) * 100.0), 2)
    else:
        metrics["OD_Center_Err_Pct"] = np.nan
        metrics["OD_Full_Crop_Pct"] = np.nan
        metrics["OD_Fail_Rate_Pct"] = np.nan

    return metrics, pred_df

def run_experiment(exp_name, arch, backbone, fusion_mode, loss_type, ablation_mode,
                   batch_size, epochs, lr, seed, gpu_id):
    set_seed(seed)
    device = torch.device(f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.cuda.set_device(gpu_id)

    exp_dir = os.path.join("./results", f"{exp_name}_seed{seed}")
    os.makedirs(exp_dir, exist_ok=True)
    os.makedirs("./checkpoints", exist_ok=True)
    best_ckpt_path = os.path.join("./checkpoints", f"{exp_name}_seed{seed}_best.pt")

    train_ds = FundusDataset("./data_splits/airogs_splits.csv", split="train", is_train=True, deg_prob=0.10)
    labels = train_ds.df["label"].values.astype(int)
    counts = np.bincount(labels, minlength=2)
    weights = 1.0 / np.maximum(counts[labels], 1)
    sampler = WeightedRandomSampler(weights, num_samples=len(train_ds), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=4, pin_memory=True, drop_last=True)

    val_ds = FundusDataset("./data_splits/airogs_splits.csv", split="val", is_train=False)
    test_ds = FundusDataset("./data_splits/airogs_splits.csv", split="test", is_train=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    if arch == "convnext_base":
        model = ConvNeXtBaseModel(backbone_name=backbone, pretrained=True).to(device)
    else:
        model = ConvNeXtProGlaucoGate(backbone_name=backbone, pretrained=True,
                                      fusion_mode=fusion_mode, ablation_mode=ablation_mode).to(device)

    supervise_q = (ablation_mode != "no_qsup")
    use_dro = (ablation_mode != "erm_no_dro")
    criterion = GlaucoGateTotalLoss(loss_type=loss_type, supervise_quality=supervise_q, use_group_dro=use_dro).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_val_pauc = -1.0
    for epoch in range(1, epochs + 1):
        cur_deg_p = min(0.40, 0.10 + (epoch - 1) * (0.30 / max(1, epochs - 1)))
        train_ds.set_deg_prob(cur_deg_p)

        model.train()
        t0 = time.time()
        running_loss, running_cls = 0.0, 0.0
        n_steps = len(train_loader)

        for step, batch in enumerate(train_loader, 1):
            xg = batch["global_img"].to(device, non_blocking=True)
            xh = batch["highres_img"].to(device, non_blocking=True)
            b_dev = {
                "label": batch["label"].to(device, non_blocking=True),
                "pseudo_domain": batch["pseudo_domain"].to(device, non_blocking=True),
                "gt_qmap": batch["gt_qmap"].to(device, non_blocking=True),
                "saliency_prior": batch["saliency_prior"].to(device, non_blocking=True)
            }
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda" if device.type == "cuda" else "cpu", enabled=(device.type == "cuda"), dtype=torch.bfloat16):
                out = model(xg, xh) if arch == "convnext_pro" else model(xg)
                loss, loss_dict = criterion(out, b_dev)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_loss += loss_dict["loss_total"]
            running_cls += loss_dict["loss_cls"]
            if step % 100 == 0 or step == n_steps:
                print(f"[{exp_name}|GPU{gpu_id}] [Epoch {epoch:02d}/{epochs:02d} | Step {step:04d}/{n_steps:04d}] "
                      f"TotalLoss={running_loss/step:.4f} (Cls={running_cls/step:.4f}) | Elapsed={time.time()-t0:.1f}s", flush=True)

        scheduler.step()
        val_m, _ = evaluate_loader(model, val_loader, device)
        print(f"[{exp_name} | Epoch {epoch:02d}/{epochs:02d} DONE] TrainLoss={running_loss/n_steps:.4f} | "
              f"Val AUC={val_m['AUC']:.4f} | Val pAUC={val_m['pAUC_90_100']:.4f} | "
              f"Val Sens@95Sp={val_m['ROC_Sens@95Sp']:.4f} | Val ECE={val_m['ECE']:.4f}", flush=True)

        if val_m["pAUC_90_100"] > best_val_pauc:
            best_val_pauc = val_m["pAUC_90_100"]
            torch.save({"model_state": model.state_dict(), "best_val_pauc": best_val_pauc, "epoch": epoch}, best_ckpt_path)

    ckpt_load = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(ckpt_load["model_state"])

    val_metrics, _ = evaluate_loader(
        model, val_loader, device,
        save_csv_path=os.path.join(exp_dir, "preds_airogs_val.csv")
    )
    tau95 = val_metrics["Frozen_Threshold"]

    summary_rows = [{"Experiment": exp_name, "Seed": seed, "Eval_Set": "AIROGS_Val", **val_metrics}]

    test_metrics, _ = evaluate_loader(
        model, test_loader, device,
        save_csv_path=os.path.join(exp_dir, "preds_airogs_test.csv"),
        fixed_threshold=tau95
    )
    summary_rows.append({"Experiment": exp_name, "Seed": seed, "Eval_Set": "AIROGS_Test", **test_metrics})

    for ext_name, ext_csv in [("PAPILA", "./data_splits/papila_external.csv"), ("Chaksu", "./data_splits/chaksu_external.csv")]:
        if os.path.exists(ext_csv):
            ext_ds = FundusDataset(ext_csv, split=None, is_train=False)
            ext_loader = DataLoader(ext_ds, batch_size=batch_size, shuffle=False, num_workers=4)

            ext_m, _ = evaluate_loader(
                model, ext_loader, device,
                save_csv_path=os.path.join(exp_dir, f"preds_{ext_name.lower()}_zeroshot.csv"),
                fixed_threshold=tau95
            )
            summary_rows.append({"Experiment": exp_name, "Seed": seed, "Eval_Set": f"{ext_name}_ZeroShot", **ext_m})

            if arch == "convnext_pro":
                oracle_m, _ = evaluate_loader(
                    model, ext_loader, device,
                    save_csv_path=os.path.join(exp_dir, f"preds_{ext_name.lower()}_oracle_crop.csv"),
                    fixed_threshold=tau95, use_oracle_crop=True
                )
                summary_rows.append({"Experiment": exp_name, "Seed": seed, "Eval_Set": f"{ext_name}_OracleCrop", **oracle_m})

                center_m, _ = evaluate_loader(
                    model, ext_loader, device,
                    save_csv_path=os.path.join(exp_dir, f"preds_{ext_name.lower()}_center_crop.csv"),
                    fixed_threshold=tau95, force_center_crop=True
                )
                summary_rows.append({"Experiment": exp_name, "Seed": seed, "Eval_Set": f"{ext_name}_FixedCenterCrop", **center_m})

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(exp_dir, "metrics_table.csv"), index=False)
    print(f"\n=== COMPLETED {exp_name} (Seed {seed}) ===", flush=True)
    print(summary_df.to_string(index=False), flush=True)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--exp_name", required=True)
    parser.add_argument("--arch", choices=["convnext_base", "convnext_pro"], default="convnext_pro")
    parser.add_argument("--backbone", default="convnext_base.fb_in22k_ft_in1k")
    parser.add_argument("--fusion_mode", default="multiplicative")
    parser.add_argument("--loss_type", default="memory_pauc_bce")
    parser.add_argument("--ablation_mode", default="full")
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--epochs", type=int, default=12)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--gpu_id", type=int, default=0)
    args = parser.parse_args()

    run_experiment(
        args.exp_name, args.arch, args.backbone, args.fusion_mode,
        args.loss_type, args.ablation_mode, args.batch_size, args.epochs,
        args.lr, args.seed, args.gpu_id
    )